# 05_random_forest.ipynb

Converted from your cell-by-cell rebuild guide.

## Cell 1 — Imports

In [1]:
import pandas as pd
import numpy as np
import joblib, json
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import (classification_report, f1_score,
                             precision_recall_curve, auc, roc_auc_score)
from imblearn.over_sampling import SMOTE
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

COINS   = ["Bitcoin", "Ethereum", "Dogecoin"]
results = {}

def load_feature_cols(coin: str):
    path = f"feature_cols_{coin.lower()}.csv"
    return pd.read_csv(path)["feature"].tolist()

def time_split(df, val_frac=0.15, test_frac=0.15):
    n = len(df)
    return (df.iloc[:int(n*(1-val_frac-test_frac))].copy(),
            df.iloc[int(n*(1-val_frac-test_frac)):int(n*(1-test_frac))].copy(),
            df.iloc[int(n*(1-test_frac)):].copy())

def evaluate(name, coin, y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    pr_auc = auc(rec, prec)
    roc    = roc_auc_score(y_true, y_prob)
    f1     = f1_score(y_true, y_pred)
    print(f"\n{'='*50}\n{name} | {coin}")
    print(classification_report(y_true, y_pred, target_names=["Down","Up"]))
    print(f"ROC-AUC: {roc:.4f}   PR-AUC: {pr_auc:.4f}   F1(Up): {f1:.4f}")
    return {"model":name,"coin":coin,"f1_up":f1,"roc_auc":roc,"pr_auc":pr_auc,"threshold":threshold}


## Cell 2 — Train Random Forest per coin

In [2]:
rf_models = {}

for coin in COINS:
    feat_cols = load_feature_cols(coin)
    df = pd.read_csv(f"features_{coin.lower()}.csv", parse_dates=["date"]).sort_values("date")
    train, val, test = time_split(df)

    X_train = train[feat_cols].values;  y_train = train["target"].values.astype(int)
    X_val   = val[feat_cols].values;    y_val   = val["target"].values.astype(int)
    X_test  = test[feat_cols].values;   y_test  = test["target"].values.astype(int)

    smote = SMOTE(random_state=42)
    X_sm, y_sm = smote.fit_resample(X_train, y_train)

    param_dist = {
        "n_estimators": [100, 200, 400],
        "max_depth": [4, 6, 8, None],
        "min_samples_leaf": [1, 5, 10],
        "max_features": ["sqrt", 0.5],
        "class_weight": ["balanced", None],
    }
    tscv = TimeSeriesSplit(n_splits=4)
    rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)
    rs = RandomizedSearchCV(
        rf_base, param_dist, n_iter=20, cv=tscv,
        scoring="f1", random_state=42, refit=True, n_jobs=-1
    )
    rs.fit(X_sm, y_sm)
    best_rf = rs.best_estimator_
    print(f"{coin} best params: {rs.best_params_} | features={len(feat_cols)}")

    # Threshold tuning on val
    val_prob = best_rf.predict_proba(X_val)[:, 1]
    best_thr = max(
        np.linspace(0.3, 0.7, 41),
        key=lambda t: f1_score(y_val, (val_prob >= t).astype(int))
    )

    test_prob = best_rf.predict_proba(X_test)[:, 1]
    res = evaluate("Random Forest", coin, y_test, test_prob, threshold=best_thr)
    results[f"rf_{coin}"] = res
    rf_models[coin] = {"model": best_rf, "threshold": best_thr, "feature_cols": feat_cols}

print("✅ Random Forest complete.")


Bitcoin best params: {'n_estimators': 100, 'min_samples_leaf': 5, 'max_features': 0.5, 'max_depth': 4, 'class_weight': 'balanced'} | features=60

Random Forest | Bitcoin
              precision    recall  f1-score   support

        Down       0.00      0.00      0.00       277
          Up       0.54      1.00      0.70       329

    accuracy                           0.54       606
   macro avg       0.27      0.50      0.35       606
weighted avg       0.29      0.54      0.38       606

ROC-AUC: 0.5095   PR-AUC: 0.5557   F1(Up): 0.7037
Ethereum best params: {'n_estimators': 100, 'min_samples_leaf': 10, 'max_features': 0.5, 'max_depth': None, 'class_weight': 'balanced'} | features=51

Random Forest | Ethereum
              precision    recall  f1-score   support

        Down       0.38      0.04      0.07       123
          Up       0.58      0.95      0.72       174

    accuracy                           0.58       297
   macro avg       0.48      0.50      0.40       297
weigh

## Cell 3 — Feature importance plot

In [3]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, coin in zip(axes, COINS):
    model = rf_models[coin]["model"]
    feat_cols = rf_models[coin]["feature_cols"]
    importances = pd.Series(model.feature_importances_, index=feat_cols)
    top15 = importances.nlargest(15).sort_values()
    top15.plot.barh(ax=ax, color="#22c55e", edgecolor="white")
    ax.set_title(f"{coin} — RF Feature Importances (Top 15)")
    ax.set_xlabel("Importance")
plt.suptitle("Random Forest Feature Importances", fontsize=14)
plt.tight_layout()
plt.savefig("rf_feature_importances.png", dpi=150)
plt.show()


## Cell 4 — Save RF models

In [4]:
for coin in COINS:
    joblib.dump(rf_models[coin]["model"], f"rf_model_{coin.lower()}.joblib")
    print(f"Saved rf_model_{coin.lower()}.joblib  threshold={rf_models[coin]['threshold']:.2f}")

Saved rf_model_bitcoin.joblib  threshold=0.30
Saved rf_model_ethereum.joblib  threshold=0.34
Saved rf_model_dogecoin.joblib  threshold=0.31
